# Borton dataset: name → SMILES (PubChem) + RDKit validation — Kaggle version

**Before running (Kaggle notebook settings, right panel):**
1. **Accelerator:** GPU T4 x2 (or P100).
2. **Internet:** On (required for PubChem; Kaggle asks for phone verification to enable it).
3. **Add Input:** upload `Borton1.xlsx` as a Kaggle Dataset and attach it. The notebook finds it under `/kaggle/input/` automatically.
4. *(Optional, to resume)* attach a previous `lookup_checkpoint.csv` as a dataset too. Already-finished rows are skipped.

**Notes**
- The PubChem lookup is network-bound; the GPU does not speed it up. Speed comes from running parallel requests while staying under PubChem's limit (≤5 requests/second).
- Files in `/kaggle/working/` are lost when an interactive session ends. Download the checkpoint, or use **Save Version → Save & Run All** so outputs are kept.
- Validation (Section 5) runs on whatever has been looked up so far, so a validated subset can be exported before all ~18k rows are done.

In [1]:
!pip -q install rdkit odfpy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 65.8 MB/s eta 0:00:00


## 1. Environment check

In [2]:
import subprocess, sys
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i))
except ImportError:
    print("PyTorch not installed")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi not available (GPU not enabled?)")

import requests
try:
    r = requests.get("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/2244/property/MolecularFormula/TXT", timeout=15)
    print("PubChem reachable:", r.status_code == 200, r.text.strip())
except Exception as e:
    print("PubChem NOT reachable. Turn Internet On in notebook settings.\n", e)

PyTorch: 2.11.0+cu128
CUDA available: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
GPU 0: Tesla T4 (UUID: GPU-20fd3af9-00a7-4ac0-865f-4329e6142058)
GPU 1: Tesla T4 (UUID: GPU-315d7137-94dc-995e-c3f2-3874b187bf35)

PubChem reachable: True C9H8O4


## 2. Configuration

In [3]:
import glob, os

# List everything attached under /kaggle/input so the right file can be picked
all_inputs = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p))
print("Files under /kaggle/input:")
for p in all_inputs:
    print("  ", p)

# Set this to the exact path printed above if auto-detection picks the wrong file
INPUT_FILE = None

if INPUT_FILE is None:
    data_files = [p for p in all_inputs
                  if p.lower().endswith((".xlsx", ".xls", ".ods", ".csv"))
                  and "checkpoint" not in os.path.basename(p).lower()]
    borton = [p for p in data_files if "borton" in os.path.basename(p).lower()]
    INPUT_FILE = (borton or data_files or [None])[0]
print("\nInput file:", INPUT_FILE)

prev_ckpts = [p for p in all_inputs if "lookup_checkpoint" in os.path.basename(p).lower()]
RESUME_FROM = prev_ckpts[0] if prev_ckpts else None
print("Resume from:", RESUME_FROM)

OUT_DIR = "/kaggle/working"
CHECKPOINT = os.path.join(OUT_DIR, "lookup_checkpoint.csv")

NAME_COL = "Chemical Name"
CAS_COL = None            # auto-detected below (column containing 'CAS')

MAX_ROWS = None           # e.g. 3000 to process only the first 3000 rows first; None = all
MAX_WORKERS = 4           # parallel requests
REQS_PER_SEC = 4.0        # stay under PubChem's 5 req/s limit
MAX_RETRIES = 5
CHECKPOINT_EVERY = 500    # rows between checkpoint saves
USE_CAS_FALLBACK = True   # if the name is not found, try each valid CAS number
RETRY_TRANSIENT = True    # on resume, re-run rows that failed for network/server reasons

Files under /kaggle/input:
   /kaggle/input/datasets/morty3500/borton/Borton1.xlsx

Input file: /kaggle/input/datasets/morty3500/borton/Borton1.xlsx
Resume from: None


## 3. Load data and parse CAS numbers

In [4]:
import re
import pandas as pd

if INPUT_FILE is None:
    raise FileNotFoundError("No input file found under /kaggle/input. Attach the dataset (Add Input).")

def read_raw(path):
    # Return {sheet_name: DataFrame} with no header applied
    p = path.lower()
    if p.endswith(".csv"):
        return {"csv": pd.read_csv(path, header=None, dtype=str)}
    engine = "odf" if p.endswith(".ods") else None
    return pd.read_excel(path, sheet_name=None, header=None, dtype=str, engine=engine)

def find_table(sheets, name_col, scan_rows=50):
    # Find the sheet and row where the header containing name_col is
    target = name_col.strip().lower()
    for sheet, raw in sheets.items():
        for i in range(min(scan_rows, len(raw))):
            vals = [str(v).strip().lower() for v in raw.iloc[i].tolist()]
            if target in vals:
                tbl = raw.iloc[i + 1:].copy()
                tbl.columns = [str(c).strip() for c in raw.iloc[i].tolist()]
                tbl = tbl.loc[:, [c for c in tbl.columns if c and c.lower() != "nan"]]
                tbl = tbl.dropna(how="all")
                return sheet, i, tbl
    return None, None, None

sheets = read_raw(INPUT_FILE)
sheet, header_row, df = find_table(sheets, NAME_COL)
if df is None:
    for sh, raw in sheets.items():
        print(f"--- sheet '{sh}': shape {raw.shape} ---")
        print(raw.head(10).to_string())
    raise ValueError(f"No '{NAME_COL}' header found in any sheet of {INPUT_FILE}. "
                     "See the preview above; set INPUT_FILE / NAME_COL in Section 2.")
print(f"Using sheet '{sheet}', header on row {header_row + 1}")

# match the name column case-insensitively
NAME_COL = next(c for c in df.columns if c.lower() == NAME_COL.strip().lower())
if CAS_COL is None:
    CAS_COL = next((c for c in df.columns if "CAS" in str(c)), None)

df = df.reset_index(drop=True)
df.insert(0, "row_id", df.index)
if MAX_ROWS:
    df = df.iloc[:MAX_ROWS].copy()

CAS_RE = re.compile(r"^([1-9]\d{1,6})-(\d{2})-(\d)$")

def cas_is_valid(cas: str) -> bool:
    """Format + check-digit test. Placeholder IDs like '001-01-2' fail (leading zero / bad checksum)."""
    m = CAS_RE.match(cas)
    if not m:
        return False
    digits = (m.group(1) + m.group(2))[::-1]
    return sum((i + 1) * int(d) for i, d in enumerate(digits)) % 10 == int(m.group(3))

def split_cas(cell):
    if pd.isna(cell):
        return []
    parts = [p.strip() for p in re.split(r"[,;\s]+", str(cell)) if p.strip()]
    return [p for p in dict.fromkeys(parts) if cas_is_valid(p)]   # dedupe, keep order

df["Valid_CAS"] = df[CAS_COL].apply(split_cas) if CAS_COL else [[] for _ in range(len(df))]

print("Rows:", len(df), "| Name column:", NAME_COL, "| CAS column:", CAS_COL)
print("Rows with at least one checksum-valid CAS:", (df["Valid_CAS"].str.len() > 0).sum())
df.head()

Using sheet 'Extractables', header on row 1
Rows: 18266 | Name column: Chemical Name | CAS column: Chemical Abstracts Service Registry Numbers (CASRN) or Other Identifyig Value
Rows with at least one checksum-valid CAS: 12242


,row_id,Chemical Name,Chemical Abstracts Service Registry Numbers (CASRN) or Other Identifyig Value,Valid_CAS
0,0,"Heptane, 3,3,5-trimethyl",7154-80-5,[7154-80-5]
1,1,"2,6-di-tert-butyl-2,5-cyclohexadiene-1,4-dione",719-22-2,[719-22-2]
2,2,Pyridinylpiperazine,"001-01-2, 34803-66-2",[34803-66-2]
3,3,"2,6-Di-tert-butyl-4-methylphenol","001-01-3, 1219805-92-1, 1-10-8, 128-37-0, 001-...","[1219805-92-1, 128-37-0]"
4,4,Mexenone,"001-01-4, 1641-17-4",[1641-17-4]


## 4. PubChem lookup (parallel, rate-limited, checkpointed, resumable)

In [5]:
import time, random, threading
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import requests

BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PROPS = "IsomericSMILES,CanonicalSMILES"

class RateLimiter:
    def __init__(self, rate):
        self.interval = 1.0 / rate
        self.lock = threading.Lock()
        self.next_t = time.monotonic()
    def wait(self):
        with self.lock:
            now = time.monotonic()
            if self.next_t > now:
                time.sleep(self.next_t - now)
            self.next_t = max(now, self.next_t) + self.interval

limiter = RateLimiter(REQS_PER_SEC)
_local = threading.local()
def session():
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
    return _local.s

def _get_json(url):
    """Returns (status_label, json_or_None)."""
    for attempt in range(MAX_RETRIES):
        limiter.wait()
        try:
            r = session().get(url, timeout=30)
        except requests.exceptions.RequestException:
            time.sleep(1.5 ** attempt + random.random())
            continue
        if r.status_code == 200:
            return "ok", r.json()
        if r.status_code == 404:
            return "not_found", None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** attempt + random.random())
            continue
        return f"HTTP {r.status_code}", None
    return "Failed after retries", None

def _pick(rec, *keys):
    # PubChem has changed SMILES property key names over time; accept either form
    for k in keys:
        if rec.get(k):
            return rec[k]
    return None

def _query(namespace_value):
    url = f"{BASE_URL}/compound/name/{quote(namespace_value, safe='')}/property/{PROPS}/JSON"
    status, data = _get_json(url)
    if status != "ok":
        return status, None
    props = data.get("PropertyTable", {}).get("Properties", [])
    if not props:
        return "not_found", None
    rec = props[0]
    return "ok", {
        "PubChem_CID": rec.get("CID"),
        "Isomeric_SMILES": _pick(rec, "IsomericSMILES", "SMILES"),
        "Canonical_SMILES": _pick(rec, "CanonicalSMILES", "ConnectivitySMILES"),
        "PubChem_Hits": len(props),
    }

EMPTY = {"PubChem_CID": None, "Isomeric_SMILES": None, "Canonical_SMILES": None, "PubChem_Hits": 0}

def lookup(row_id, name, cas_list):
    out = {"row_id": row_id, "Matched_By": None, "Matched_Query": None, **EMPTY}
    queries = []
    if isinstance(name, str) and name.strip():
        queries.append(("name", name.strip()))
    if USE_CAS_FALLBACK:
        queries += [("CAS", c) for c in cas_list]
    if not queries:
        out["Status"] = "Empty name / no valid CAS"
        return out
    last_status = "not_found"
    for kind, q in queries:
        status, rec = _query(q)
        if status == "ok":
            out.update(rec, Matched_By=kind, Matched_Query=q, Status="Success")
            return out
        last_status = status
        if status not in ("not_found",) and not status.startswith("HTTP 4"):
            break   # transient/server problem: don't burn more queries on this row
    out["Status"] = "Not found in PubChem" if last_status == "not_found" or last_status.startswith("HTTP 4") else last_status
    return out

TRANSIENT = lambda s: isinstance(s, str) and (s.startswith("Failed") or s.startswith("HTTP 5") or s.startswith("Error"))

# ---- resume ----
results = {}
for src in [CHECKPOINT, RESUME_FROM]:
    if src and os.path.exists(src):
        prev = pd.read_csv(src)
        for rec in prev.to_dict("records"):
            if RETRY_TRANSIENT and TRANSIENT(rec.get("Status")):
                continue
            results.setdefault(int(rec["row_id"]), rec)
print("Already done:", len(results))

todo = df[~df["row_id"].isin(results.keys())]
print("To query:", len(todo))

def save_checkpoint():
    tmp = CHECKPOINT + ".tmp"
    pd.DataFrame(list(results.values())).sort_values("row_id").to_csv(tmp, index=False)
    os.replace(tmp, CHECKPOINT)

rows = list(todo[["row_id", NAME_COL, "Valid_CAS"]].itertuples(index=False, name=None))
with tqdm(total=len(rows), desc="PubChem") as bar:
    for start in range(0, len(rows), CHECKPOINT_EVERY):
        chunk = rows[start:start + CHECKPOINT_EVERY]
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = [ex.submit(lookup, rid, nm, cas) for rid, nm, cas in chunk]
            for f in as_completed(futs):
                try:
                    r = f.result()
                except Exception as e:
                    continue   # row stays missing -> retried on next run
                results[r["row_id"]] = r
                bar.update(1)
        save_checkpoint()

save_checkpoint()
print("Checkpoint:", CHECKPOINT, "| rows:", len(results))
pd.Series([r["Status"] for r in results.values()]).value_counts()

Already done: 0
To query: 18266


PubChem:   0%|          | 0/18266 [00:00<?, ?it/s]

Checkpoint: /kaggle/working/lookup_checkpoint.csv | rows: 18266


Success                 12015
Not found in PubChem     5894
Failed after retries      357
Name: count, dtype: int64

## 5. RDKit structure validation

Works on whatever is in the checkpoint, including a partial run.

Rows are **flagged, not removed**, for: multi-fragment structures (salts/mixtures), duplicate structures (same InChIKey), and rows matched by CAS fallback instead of name. Whether to keep those is a curation decision.

In [6]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.DisableLog("rdApp.*")

def validate(smi):
    base = {"RDKit_Valid": False, "RDKit_SMILES": None, "InChIKey": None,
            "Num_Fragments": None, "Heavy_Atoms": None, "MolWt": None}
    if not isinstance(smi, str) or not smi.strip():
        return base
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return base
    return {"RDKit_Valid": True,
            "RDKit_SMILES": Chem.MolToSmiles(mol),
            "InChIKey": Chem.MolToInchiKey(mol) or None,
            "Num_Fragments": len(Chem.GetMolFrags(mol)),
            "Heavy_Atoms": mol.GetNumHeavyAtoms(),
            "MolWt": round(Descriptors.MolWt(mol), 3)}

res_df = pd.DataFrame(list(results.values()))
for col in ["Isomeric_SMILES", "Canonical_SMILES"]:
    if col not in res_df.columns:
        res_df[col] = None
res_df["SMILES_used"] = res_df["Isomeric_SMILES"].where(res_df["Isomeric_SMILES"].notna(), res_df["Canonical_SMILES"])
val = pd.DataFrame([validate(s) for s in tqdm(res_df["SMILES_used"], desc="RDKit")])
res_df = pd.concat([res_df.reset_index(drop=True), val], axis=1)

full = df.drop(columns=["Valid_CAS"]).merge(res_df, on="row_id", how="left")
full["Flag_MultiFragment"] = full["Num_Fragments"].fillna(1) > 1
full["Flag_Duplicate_Structure"] = full["InChIKey"].notna() & full.duplicated("InChIKey", keep="first")
full["Flag_Matched_By_CAS"] = full["Matched_By"].eq("CAS")
full["Validated"] = full["Status"].eq("Success") & full["RDKit_Valid"].fillna(False).astype(bool)

print("Looked up:", full["Status"].notna().sum(), "/", len(full))
print("Validated (PubChem hit + RDKit-parsable):", int(full["Validated"].sum()))
print("  of which multi-fragment:", int((full["Validated"] & full["Flag_MultiFragment"]).sum()))
print("  of which duplicate structures:", int((full["Validated"] & full["Flag_Duplicate_Structure"]).sum()))
print("  of which matched by CAS fallback:", int((full["Validated"] & full["Flag_Matched_By_CAS"]).sum()))
full["Status"].value_counts(dropna=False)

RDKit:   0%|          | 0/18266 [00:00<?, ?it/s]

Looked up: 18266 / 18266
Validated (PubChem hit + RDKit-parsable): 12015
  of which multi-fragment: 171
  of which duplicate structures: 1006
  of which matched by CAS fallback: 2490


Status
Success                 12015
Not found in PubChem     5894
Failed after retries      357
Name: count, dtype: int64

## 6. Export

In [7]:
validated = full[full["Validated"]].copy()

full.to_csv(os.path.join(OUT_DIR, "Borton_SMILES_all_rows.csv"), index=False)
validated.to_csv(os.path.join(OUT_DIR, "Borton_SMILES_validated.csv"), index=False)
with pd.ExcelWriter(os.path.join(OUT_DIR, "Borton_SMILES.xlsx")) as xw:
    validated.to_excel(xw, sheet_name="validated", index=False)
    full.to_excel(xw, sheet_name="all_rows", index=False)
    full[~full["Validated"] & full["Status"].notna()].to_excel(xw, sheet_name="needs_review", index=False)

print(f"Saved to {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" ", f)
validated.head(10)

Saved to /kaggle/working:
  Borton_SMILES.xlsx
  Borton_SMILES_all_rows.csv
  Borton_SMILES_validated.csv
  __notebook__.ipynb
  lookup_checkpoint.csv


,row_id,Chemical Name,Chemical Abstracts Service Registry Numbers (CASRN) or Other Identifyig Value,Matched_By,Matched_Query,PubChem_CID,Isomeric_SMILES,Canonical_SMILES,PubChem_Hits,Status,...,RDKit_Valid,RDKit_SMILES,InChIKey,Num_Fragments,Heavy_Atoms,MolWt,Flag_MultiFragment,Flag_Duplicate_Structure,Flag_Matched_By_CAS,Validated
0,0,"Heptane, 3,3,5-trimethyl",7154-80-5,CAS,7154-80-5,23544.0,CCC(C)CC(C)(C)CC,CCC(C)CC(C)(C)CC,1,Success,...,True,CCC(C)CC(C)(C)CC,VRVRZZWPKABUOE-UHFFFAOYSA-N,1.0,10.0,142.286,False,False,True,True
1,1,"2,6-di-tert-butyl-2,5-cyclohexadiene-1,4-dione",719-22-2,CAS,719-22-2,12867.0,CC(C)(C)C1=CC(=O)C=C(C1=O)C(C)(C)C,CC(C)(C)C1=CC(=O)C=C(C1=O)C(C)(C)C,1,Success,...,True,CC(C)(C)C1=CC(=O)C=C(C(C)(C)C)C1=O,RDQSIADLBQFVMY-UHFFFAOYSA-N,1.0,16.0,220.312,False,False,True,True
2,2,Pyridinylpiperazine,"001-01-2, 34803-66-2",name,Pyridinylpiperazine,94459.0,C1CN(CCN1)C2=CC=CC=N2,C1CN(CCN1)C2=CC=CC=N2,1,Success,...,True,c1ccc(N2CCNCC2)nc1,GZRKXKUVVPSREJ-UHFFFAOYSA-N,1.0,12.0,163.224,False,False,False,True
3,3,"2,6-Di-tert-butyl-4-methylphenol","001-01-3, 1219805-92-1, 1-10-8, 128-37-0, 001-...",name,"2,6-Di-tert-butyl-4-methylphenol",31404.0,CC1=CC(=C(C(=C1)C(C)(C)C)O)C(C)(C)C,CC1=CC(=C(C(=C1)C(C)(C)C)O)C(C)(C)C,1,Success,...,True,Cc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1,NLZUEZXRPGMBCV-UHFFFAOYSA-N,1.0,16.0,220.356,False,False,False,True
4,4,Mexenone,"001-01-4, 1641-17-4",name,Mexenone,71645.0,CC1=CC=C(C=C1)C(=O)C2=C(C=C(C=C2)OC)O,CC1=CC=C(C=C1)C(=O)C2=C(C=C(C=C2)OC)O,1,Success,...,True,COc1ccc(C(=O)c2ccc(C)cc2)c(O)c1,MJVGBKJNTFCUJM-UHFFFAOYSA-N,1.0,18.0,242.274,False,False,False,True
5,5,Diphenylphosphine oxide,"001-01-4, 4559-70-0",CAS,4559-70-0,254003.0,C1=CC=C(C=C1)P(=O)C2=CC=CC=C2,C1=CC=C(C=C1)P(=O)C2=CC=CC=C2,1,Success,...,True,O=[PH](c1ccccc1)c1ccccc1,ASUOLLHGALPRFK-UHFFFAOYSA-N,1.0,14.0,202.193,False,False,True,True
6,6,Cyasorb 5411,"001-01-6, 123307-21-1, 52188-76-8, 3147-75-9",CAS,123307-21-1,62485.0,CC(C)(C)CC(C)(C)C1=CC(=C(C=C1)O)N2N=C3C=CC=CC3=N2,CC(C)(C)CC(C)(C)C1=CC(=C(C=C1)O)N2N=C3C=CC=CC3=N2,1,Success,...,True,CC(C)(C)CC(C)(C)c1ccc(O)c(-n2nc3ccccc3n2)c1,IYAZLDLPUNDVAG-UHFFFAOYSA-N,1.0,24.0,323.440,False,False,True,True
7,7,2-tert-Butyl-4-methylphenol (BHT fragment),"001-01-7, 2409-55-4, 4-06-0, 25567-40-2, 29759...",CAS,2409-55-4,17004.0,CC1=CC(=C(C=C1)O)C(C)(C)C,CC1=CC(=C(C=C1)O)C(C)(C)C,1,Success,...,True,Cc1ccc(O)c(C(C)(C)C)c1,IKEHOXWJQXIQAG-UHFFFAOYSA-N,1.0,12.0,164.248,False,False,True,True
8,8,4-(p-Tolylthio)benzophenone,"001-01-7, 83846-85-9, 26960-79-2",name,4-(p-Tolylthio)benzophenone,174341.0,CC1=CC=C(C=C1)SC2=CC=C(C=C2)C(=O)C3=CC=CC=C3,CC1=CC=C(C=C1)SC2=CC=C(C=C2)C(=O)C3=CC=CC=C3,1,Success,...,True,Cc1ccc(Sc2ccc(C(=O)c3ccccc3)cc2)cc1,DBHQYYNDKZDVTN-UHFFFAOYSA-N,1.0,22.0,304.414,False,False,False,True
9,9,"4,4'-Butylidenebis(6-tert-butyl-m-cresol)","001-01-8, 4-06-0, 85-60-9",name,"4,4'-Butylidenebis(6-tert-butyl-m-cresol)",6815.0,CCCC(C1=CC(=C(C=C1C)O)C(C)(C)C)C2=CC(=C(C=C2C)...,CCCC(C1=CC(=C(C=C1C)O)C(C)(C)C)C2=CC(=C(C=C2C)...,1,Success,...,True,CCCC(c1cc(C(C)(C)C)c(O)cc1C)c1cc(C(C)(C)C)c(O)...,PFANXOISJYKQRP-UHFFFAOYSA-N,1.0,28.0,382.588,False,False,False,True


## 7. Modeling

The GPU is used from here on. Add model cells once the target property and evaluation split are agreed, reading from `validated` (or `Borton_SMILES_validated.csv`).